# Hands-on 2 — Toy MC: reweighting, effective area and PSF

**PhD school - STUDENT EXERCISES**  
Built to match the ideas in lecture **slide 24** (raw $E^{-1}$ MC reweighted to $E^{-2}$, $E^{-2.7}$, $E^{-3.8}$), plus **slides 56–81** (area, energy resolution, angular resolution, PSF).

We simulate a **log-uniform neutrino generation spectrum** and later a deliberately simplified detector. The detector and fluxes are **illustrative**, not IceCube predictions. There is no external download.

**Learning goals:** understand density transformations; reweight one MC sample to several power laws; make a toy selection; derive an area curve and conditional energy response; derive a *genuine* angular separation/PSF using both zenith and azimuth.

## A. Generate convenient MC, reweight to physics

For generated \(q(E)\propto E^{-1}\), \(dN/d\log_{10}E=\ln(10)E\,dN/dE\) is constant. A physical \(E^{-\gamma}\) shape is recovered by weights proportional to \(\Phi_\gamma(E)/q(E)\propto E^{1-\gamma}\). Here the curves are **pivot-normalized at 10 TeV**, but have arbitrary amplitude, i.e. **not** an absolute detector rate.

The lecture's slide 24 is a conceptual histogram where the unweighted sample is flat and the reweighted spectra intersect at the pivot.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize':(8,4.8),'axes.grid':True,'grid.alpha':.23})
rng = np.random.default_rng(2026)
N_GENERATED = 160_000
E_MIN, E_MAX, E_PIVOT = 1e2, 1e7, 1e4  # GeV
GAMMAS = [2.0, 2.7, 3.8]
# Inverse-CDF sampling for q(E) \propto E^-1 over finite limits.
E = 10**rng.uniform(np.log10(E_MIN), np.log10(E_MAX), N_GENERATED)
log_bins = np.logspace(np.log10(E_MIN),np.log10(E_MAX),30)

def relative_weight(energy, gamma):
    """Relative to q(E) ∝ E^-1; pivot-normalized weight (arbitrary units)."""
    return (energy/E_PIVOT)**(1-gamma)

def weighted_quantile(x, w, quantiles):
    x=np.asarray(x,float);w=np.asarray(w,float)
    good=np.isfinite(x)&np.isfinite(w)&(w>0)
    if not np.any(good):return np.full(len(quantiles),np.nan)
    index=np.argsort(x[good]);a=x[good][index];b=w[good][index]
    return np.interp(quantiles,(np.cumsum(b)-b/2)/b.sum(),a)

print('Generated MC:',len(E),' / E range:',(E.min(),E.max()))

Generated MC: 160000  / E range: (100.00528087936515, 9999531.550022429)


### Exercise 1 — reproduce lecture slide 24

1. Check numerically that raw MC is flat in equal \(\log_{10}E\) bins, not equal linear-energy bins.
2. Plot raw MC in black and pivot-normalized weighted MC for \(\gamma=2,2.7,3.8\) in the same **log-energy bins**, with logarithmic y axis. Make the curves cross near 10 TeV.
3. Show mathematically: why is the slope in a plot of \(dN/d\log_{10}E\) proportional to \(E^{1-\gamma}\), rather than \(E^{-\gamma}\)?
4. Adjust pivot and indices; distinguish a weighted **event count per bin** from a physical differential flux.

In [ ]:
# TODO: write your code here.
# Use the named variables/functions from the setup cells.

### Exercise 2 — play with weights and expected yields

1. Compare the weighted totals for different \(\gamma\), holding the value at the **pivot** fixed. Split totals below and above the pivot.
2. Set the generation range to only \(E>E_{\rm pivot}\) conceptually: which gamma now gives the biggest yield and why?
3. Multiply all weights by two: what changes and what stays the same? Try a normalization \(\Phi_0\) chosen at a different pivot.
4. Is a sum of these arbitrary relative weights a physical expected event count? What additional pieces would be required?

In [ ]:
# TODO: write your code here.
# Use the named variables/functions from the setup cells.

## B. Simulate a simple detector (known generation + selected events)

Unlike HESE public selected MC, this toy retains the **complete generated sample**. We know which events pass selection, the generation area, both true and reconstructed directions, a true interaction category and a toy true topology. Therefore selection efficiency, response and angular PSF can be calculated directly.

In [ ]:
# Toy acceptance with a smooth energy turn-on + weaker horizon sensitivity.
A_GENERATION_CM2 = 1e8  # arbitrary reference area, 10,000 m^2
coszen = rng.uniform(-1,1,len(E))
phi_true = rng.uniform(0,2*np.pi,len(E))
p_energy = .002 + .28/(1+(3e4/E)**1.2)
p_accept = p_energy*(.68 + .32*(1-coszen)/2)
selected = rng.uniform(size=len(E)) < p_accept

# Toy reco morphology: 0=cascade, 1=track, correlated with the true interaction label.
interaction = rng.choice([1,2],size=len(E),p=[.7,.3])  # 1=CC, 2=NC
topology_true = np.where(interaction==2, 0, rng.choice([0,1],len(E),p=[.55,.45]))
p_reco_track = np.where(topology_true==1,.91,.08)
topology_reco = (rng.uniform(size=len(E))<p_reco_track).astype(int)

# Reconstructed deposited energy includes random visible-energy fraction and smearing.
f_visible = np.where(interaction==1, .78, .35)
f_visible = np.where(topology_true==1, f_visible*.66, f_visible)
e_reco = E*f_visible*10**rng.normal(0,.18,size=len(E))

# Unit sphere: both zenith and azimuth of true/reconstructed directions available in toy.
th_true = np.arccos(coszen)
n_true=np.column_stack([np.sin(th_true)*np.cos(phi_true),
                        np.sin(th_true)*np.sin(phi_true), np.cos(th_true)])
e1=np.column_stack([-np.sin(phi_true),np.cos(phi_true),np.zeros(len(E))])
e2=np.cross(n_true,e1)
sigma_deg=np.where(topology_true==1,1.0,8.0)*(E/1e5)**(-.10)
sigma_rad=np.deg2rad(np.clip(sigma_deg,.25,20.0))
err1=rng.normal(size=len(E))*sigma_rad
err2=rng.normal(size=len(E))*sigma_rad
n_reco=n_true + err1[:,None]*e1 + err2[:,None]*e2
n_reco/=np.linalg.norm(n_reco,axis=1)[:,None]
psi=np.arccos(np.clip(np.sum(n_true*n_reco,axis=1),-1,1))  # RAD, full 3-D opening angle
th_reco=np.arccos(np.clip(n_reco[:,2],-1,1))

print(f'Selected toy events: {selected.sum():,} / {len(E):,} ({selected.mean():.2%})')

### Exercise 3 — effective area from known generation and selection

1. For equal \(\log E\) bins, compute \(\epsilon(E)=N_{\rm sel}(E)/N_{\rm gen}(E)\). Then use \(A_{\rm eff}(E)=A_{\rm generation}\epsilon(E)\) for the toy. What happens at the energy threshold?
2. Split into true \(\cos\theta\) bands and repeat. Why is this possible in the toy but the corresponding *raw selection efficiency* is unavailable in the HESE public selected sample?
3. Compute the **relative number of detected events** using reweighted selected events for each \(\gamma\). Contrast with raw generated reweighted totals. Does the ordering always stay the same?

In [ ]:
# TODO: write your code here.
# Use the named variables/functions from the setup cells.

### Exercise 4 — energy response and weighted bias

1. For the **selected** toy events, construct a row-normalized \(P(E_{\rm reco}\mid E_{\rm true},\mathrm{selected})\) in logarithmic energy bins.
2. Calculate \(\delta=\log_{10}(E_{\rm reco}/E_{\rm true})\) and weighted quantiles (16%, 50%, 84%) in true-energy bins; compare \(\gamma=2\) and \(3.8\).
3. Repeat separately for truth-level track/cascade toy topologies. What is *intrinsic smearing* versus *flux-weighted population mixing*?

In [ ]:
# TODO: write your code here.
# Use the named variables/functions from the setup cells.

### Exercise 5 — angular resolution and the point-spread function

1. Using the **3D unit vectors** of the generated neutrino and reconstructed event, calculate \(\Delta\psi=\arccos(\hat n_{\rm true}\cdot\hat n_{\rm reco})\). Plot it in degrees versus true energy for selected events (optional: only a random subset for legibility).
2. Obtain weighted **50% and 90% containment angles** in true-energy bins separately for reconstructed tracks and cascades. Is \(|\theta_{\rm reco}-\theta_{\rm true}|\) the same quantity?
3. Estimate the azimuthally averaged PSF \(dP/d\Omega\) as a function of angular separation \(\psi\). For annuli, use \(\Delta\Omega=2\pi(\cos\psi_{\rm low}-\cos\psi_{\rm high})\), not simply \(\Delta\psi\). Normalize and check the integral.
4. Change \(\gamma\). Which **pooled** angular distributions change, and why?

In [ ]:
# TODO: write your code here.
# Use the named variables/functions from the setup cells.

## C. Discussion

- **Slide 24:** Why generate from a non-physical spectrum and add event weights later?
- **Event rates:** What does it take to assign these toy weighted entries physical units and real exposure? How does a detector threshold change the number of detected events for a steep spectrum?
- **Detector response:** What can the toy do because it knows *all generated events* and both true/reconstructed azimuths that the 7.5-year HESE public files cannot?
- **Systematics:** Which fake features here are assumptions? How would you investigate robustness against the visible-energy fraction, angular smearing, selection threshold, and topology mixture?

**Take-home:** event weights change the physics hypothesis; response functions describe a detector/analysis selection. Population-averaged response plots can depend on event weights even when the underlying conditional detector model does not.